# 00 · Data acquisition and verification
**Study Smarter, Not Harder** · MPBA G505 group project

Source: Cortez, P. (2008). *Student Performance* [Dataset]. UCI Machine Learning Repository.
https://doi.org/10.24432/C5TG7T (CC BY 4.0). Downloaded 2026-11-05 from
`https://archive.ics.uci.edu/static/public/320/student+performance.zip` (the zip contains `student.zip`,
which holds the two CSVs). Only `student-por.csv` and `student-mat.csv` were copied into `data/raw/`;
they are read-only and must never be edited. Everything derived goes to `data/processed/`.

This notebook: loads both files, checks shape, hashes, missing values and value ranges, builds a data
dictionary, flags the G3 = 0 zero-grade cases, and saves cleaned copies.

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path("../data/raw")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 140)

## 1. Integrity of the raw files
SHA-256 hashes recorded at download time. If a hash changes, the raw file was modified.

In [2]:
EXPECTED_SHA256 = {
    "student-por.csv": "a7594a11d7771c0efe1a740824e0e833da9c4cad07c39a9766a874575563fb3f",
    "student-mat.csv": "e47f9ee225e1ee6e69b7564e6dac7123e80b8486677fe111f351964cef5dec80",
}
for fname, expected in EXPECTED_SHA256.items():
    h = hashlib.sha256((RAW / fname).read_bytes()).hexdigest()
    print(f"{fname}: {'OK' if h == expected else 'CHANGED!'} {h}")
    assert h == expected

student-por.csv: OK a7594a11d7771c0efe1a740824e0e833da9c4cad07c39a9766a874575563fb3f
student-mat.csv: OK e47f9ee225e1ee6e69b7564e6dac7123e80b8486677fe111f351964cef5dec80


## 2. Load and verify shape, missing values, duplicates

In [3]:
por = pd.read_csv(RAW / "student-por.csv", sep=";")
mat = pd.read_csv(RAW / "student-mat.csv", sep=";")
for name, df, shape in [("por", por, (649, 33)), ("mat", mat, (395, 33))]:
    print(f"{name}: shape={df.shape} (expected {shape}), missing={int(df.isna().sum().sum())}, "
          f"duplicate rows={int(df.duplicated().sum())}")
    assert df.shape == shape and df.isna().sum().sum() == 0
assert list(por.columns) == list(mat.columns)
por.head()

por: shape=(649, 33) (expected (649, 33)), missing=0, duplicate rows=0
mat: shape=(395, 33) (expected (395, 33)), missing=0, duplicate rows=0


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


## 3. Value ranges against the documented codes

In [4]:
RANGES = {"age": (15, 22), "Medu": (0, 4), "Fedu": (0, 4), "traveltime": (1, 4), "studytime": (1, 4),
          "failures": (0, 4), "famrel": (1, 5), "freetime": (1, 5), "goout": (1, 5), "Dalc": (1, 5),
          "Walc": (1, 5), "health": (1, 5), "absences": (0, 93), "G1": (0, 20), "G2": (0, 20), "G3": (0, 20)}
rows = []
for name, df in [("por", por), ("mat", mat)]:
    for col, (lo, hi) in RANGES.items():
        rows.append({"file": name, "var": col, "min": df[col].min(), "max": df[col].max(),
                     "in_range": df[col].between(lo, hi).all()})
rng = pd.DataFrame(rows)
print("All numeric values inside documented ranges:", rng.in_range.all())
rng.pivot(index="var", columns="file", values=["min", "max"])

All numeric values inside documented ranges: True


min     max    
file       mat por mat por
var                       
Dalc         1   1   5   5
Fedu         0   0   4   4
G1           3   0  19  19
G2           0   0  19  19
G3           0   0  20  19
Medu         0   0   4   4
Walc         1   1   5   5
absences     0   0  75  32
age         15  15  22  22
failures     0   0   3   3
famrel       1   1   5   5
freetime     1   1   5   5
goout        1   1   5   5
health       1   1   5   5
studytime    1   1   4   4
traveltime   1   1   4   4

In [5]:
cats = por.select_dtypes("object").columns
pd.DataFrame({c: [", ".join(sorted(por[c].unique()))] for c in cats}, index=["levels (por)"]).T

,levels (por)
school,"GP, MS"
sex,"F, M"
address,"R, U"
famsize,"GT3, LE3"
Pstatus,"A, T"
Mjob,"at_home, health, other, services, teacher"
Fjob,"at_home, health, other, services, teacher"
reason,"course, home, other, reputation"
guardian,"father, mother, other"
schoolsup,"no, yes"


## 4. Data dictionary
Descriptions from `student.txt` in the UCI zip; `type`/`levels` computed from the Portuguese file.

In [6]:
DESC = {
    "school": "student's school (GP = Gabriel Pereira, MS = Mousinho da Silveira)",
    "sex": "student's sex (F, M)", "age": "age in years (15-22)",
    "address": "home address type (U = urban, R = rural)", "famsize": "family size (LE3 = <=3, GT3 = >3)",
    "Pstatus": "parents' cohabitation (T = together, A = apart)",
    "Medu": "mother's education (0 none, 1 primary, 2 5th-9th grade, 3 secondary, 4 higher)",
    "Fedu": "father's education (same coding as Medu)",
    "Mjob": "mother's job (teacher, health, services, at_home, other)", "Fjob": "father's job (same levels as Mjob)",
    "reason": "reason for choosing school (home, reputation, course, other)",
    "guardian": "student's guardian (mother, father, other)",
    "traveltime": "home-school travel time (1 <15 min, 2 15-30 min, 3 30-60 min, 4 >1 h)",
    "studytime": "weekly study time (1 <2 h, 2 2-5 h, 3 5-10 h, 4 >10 h) - ordinal category, not hours",
    "failures": "number of past class failures (n if 1<=n<3, else 4)",
    "schoolsup": "extra educational support from school (yes/no)", "famsup": "family educational support (yes/no)",
    "paid": "extra paid classes in the subject (yes/no)", "activities": "extra-curricular activities (yes/no)",
    "nursery": "attended nursery school (yes/no)", "higher": "wants to take higher education (yes/no)",
    "internet": "internet access at home (yes/no)", "romantic": "in a romantic relationship (yes/no)",
    "famrel": "quality of family relationships (1 very bad - 5 excellent)",
    "freetime": "free time after school (1 very low - 5 very high)",
    "goout": "going out with friends (1 very low - 5 very high)",
    "Dalc": "workday alcohol consumption (1 very low - 5 very high)",
    "Walc": "weekend alcohol consumption (1 very low - 5 very high)",
    "health": "current health status (1 very bad - 5 very good)",
    "absences": "number of school absences (0-93)",
    "G1": "first-period grade (0-20)", "G2": "second-period grade (0-20)", "G3": "final grade (0-20), the outcome",
    "zero_grade": "DERIVED: True if G3 == 0 (zero-grade case; read as an apparent dropout, which is our interpretation and not recorded in the dataset; see notebooks 01 and 04)",
    "log_abs": "DERIVED: log(absences + 1)",
}
def summarise(df):
    out = []
    for c in df.columns:
        s = df[c]
        levels = (", ".join(map(str, sorted(s.unique()))) if s.nunique() <= 6
                  else f"{s.min()} to {s.max()}")
        out.append({"variable": c, "dtype": str(s.dtype), "n_unique": s.nunique(), "levels_or_range": levels,
                    "description": DESC.get(c, "")})
    return pd.DataFrame(out)

## 5. Cleaned copies with the zero-grade flag
No rows are dropped. Two columns are added: `zero_grade` (G3 == 0) and `log_abs` = log(absences + 1). Later notebooks decide whether to analyse with or without zero-grade cases.

In [7]:
clean = {}
for name, df in [("por", por), ("mat", mat)]:
    d = df.copy()
    d["zero_grade"] = d["G3"].eq(0)
    d["log_abs"] = np.log1p(d["absences"])
    clean[name] = d
    d.to_csv(PROC / f"student-{name}_clean.csv", index=False)
    print(f"{name}: saved {d.shape} -> data/processed/student-{name}_clean.csv | zero-grade cases (G3==0): {int(d.zero_grade.sum())}")

dictionary = summarise(clean["por"])
dictionary.to_csv(PROC / "data_dictionary.csv", index=False)
dictionary

por: saved (649, 35) -> data/processed/student-por_clean.csv | zero-grade cases (G3==0): 15
mat: saved (395, 35) -> data/processed/student-mat_clean.csv | zero-grade cases (G3==0): 38


,variable,dtype,n_unique,levels_or_range,description
0,school,object,2,"GP, MS","student's school (GP = Gabriel Pereira, MS = M..."
1,sex,object,2,"F, M","student's sex (F, M)"
2,age,int64,8,15 to 22,age in years (15-22)
3,address,object,2,"R, U","home address type (U = urban, R = rural)"
4,famsize,object,2,"GT3, LE3","family size (LE3 = <=3, GT3 = >3)"
5,Pstatus,object,2,"A, T","parents' cohabitation (T = together, A = apart)"
6,Medu,int64,5,"0, 1, 2, 3, 4","mother's education (0 none, 1 primary, 2 5th-9..."
7,Fedu,int64,5,"0, 1, 2, 3, 4",father's education (same coding as Medu)
8,Mjob,object,5,"at_home, health, other, services, teacher","mother's job (teacher, health, services, at_ho..."
9,Fjob,object,5,"at_home, health, other, services, teacher",father's job (same levels as Mjob)


## 6. Round-trip check
The processed file reloads identically, and raw files are unchanged.

In [8]:
re = pd.read_csv(PROC / "student-por_clean.csv")
assert re.shape == (649, 35) and re.zero_grade.sum() == clean["por"].zero_grade.sum()
for fname, expected in EXPECTED_SHA256.items():
    assert hashlib.sha256((RAW / fname).read_bytes()).hexdigest() == expected
print("processed files OK; raw files untouched")

processed files OK; raw files untouched
